# 01 — Data Collection

Fetches 15 Wikipedia articles about the 2026 FIFA World Cup and related topics,
chunks them into retrievable segments, and saves the result to `data/chunks.json`.

**Run order:** This notebook must be run **first** before any retrieval notebooks.

**Output:** `../data/chunks.json` — list of chunk dicts with keys `id`, `source`, `url`, `text`.

**Reference:** EXPERIMENT.md §Dataset for target pages and chunking strategy.

In [1]:
# Cell 1 — Install dependencies
# Install if not already present
!pip install wikipedia-api==0.6.0 tqdm

  Using cached Wikipedia_API-0.6.0-py3-none-any.whl.metadata (22 kB)


In [2]:
# Cell 2 — Imports
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.chunker import fetch_wikipedia_page, build_chunk_list, save_chunks
from tqdm import tqdm

In [3]:
# Cell 3 — Define target pages
# 15 Wikipedia articles as specified in EXPERIMENT.md §Dataset
TARGET_PAGES = [
    "2026 FIFA World Cup",
    "2026 FIFA World Cup squads",
    "FIFA World Cup records and statistics",
    "2022 FIFA World Cup",
    "Lionel Messi",
    "Kylian Mbappé",
    "Cristiano Ronaldo",
    "Erling Haaland",
    "Lamine Yamal",
    "Vinicius Jr.",
    "Jude Bellingham",
    "History of the FIFA World Cup",
    "Association football",
    "FIFA",
    "Goalkeeper (association football)"
]

print(f"Target pages: {len(TARGET_PAGES)}")
for i, p in enumerate(TARGET_PAGES, 1):
    print(f"  {i:2}. {p}")

Target pages: 15
   1. 2026 FIFA World Cup
   2. 2026 FIFA World Cup squads
   3. FIFA World Cup records and statistics
   4. 2022 FIFA World Cup
   5. Lionel Messi
   6. Kylian Mbappé
   7. Cristiano Ronaldo
   8. Erling Haaland
   9. Lamine Yamal
  10. Vinicius Jr.
  11. Jude Bellingham
  12. History of the FIFA World Cup
  13. Association football
  14. FIFA
  15. Goalkeeper (association football)


In [4]:
# Cell 4 — Fetch all pages with tqdm progress bar
pages = []
failed = []

for title in tqdm(TARGET_PAGES, desc="Fetching Wikipedia pages"):
    result = fetch_wikipedia_page(title)
    if result is not None:
        pages.append(result)
        print(f"  [OK]  {title} ({len(result['text'])} chars)")
    else:
        failed.append(title)
        print(f"  [FAIL] {title} — page not found")

print(f"\nFetched: {len(pages)} / {len(TARGET_PAGES)} pages")
if failed:
    print(f"Failed:  {failed}")

Fetching Wikipedia pages:   7%|██████▌                                                                                           | 1/15 [00:02<00:38,  2.78s/it]

  [OK]  2026 FIFA World Cup (45414 chars)


Fetching Wikipedia pages:  13%|█████████████                                                                                     | 2/15 [00:04<00:29,  2.25s/it]

  [OK]  2026 FIFA World Cup squads (11052 chars)


Fetching Wikipedia pages:  20%|███████████████████▌                                                                              | 3/15 [00:06<00:22,  1.91s/it]

...

  [OK]  Association football (47991 chars)


Fetching Wikipedia pages:  93%|██████████████████████████████████████████████████████████████████████████████████████████▌      | 14/15 [00:22<00:01,  1.41s/it]

  [OK]  FIFA (39976 chars)


Fetching Wikipedia pages: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [00:24<00:00,  1.61s/it]

  [OK]  Goalkeeper (association football) (25954 chars)

Fetched: 15 / 15 pages


In [5]:
# Cell 5 — Build chunk list and show stats
chunks = build_chunk_list(pages)

print(f"Total chunks: {len(chunks)}")
print()

# Chunks per source table
from collections import Counter
counts = Counter(c['source'] for c in chunks)
print(f"{'Source':<45} {'Chunks':>6}")
print("-" * 53)
for source, n in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"{source:<45} {n:>6}")

print()
print("Sample chunk (id=0):")
print(f"  Source: {chunks[0]['source']}")
print(f"  URL   : {chunks[0]['url']}")
print(f"  Text  : {chunks[0]['text'][:300]} ...")

Total chunks: 1918

Source                                        Chunks
-----------------------------------------------------
Cristiano Ronaldo                                246
2022 FIFA World Cup                              231
Lionel Messi                                     196
Association football                             160
2026 FIFA World Cup                              157
Erling Haaland                                   139
FIFA                                             135
History of the FIFA World Cup                    119
Kylian Mbappé                                    105
Jude Bellingham                                   93
Vinícius Júnior                                   89
Goalkeeper (association football)                 86
Lamine Yamal                                      66
2026 FIFA World Cup squads                        65
FIFA World Cup records and statistics             31

Sample chunk (id=0):
  Source: 2026 FIFA World Cup
  URL   : https://en.wikip

In [6]:
# Cell 6 — Save chunks to disk
save_chunks(chunks, "../data/chunks.json")

Saved 1918 chunks to ../data/chunks.json
